# nb950 -- chemprop_aux v2 (Kaggle T4 offload)

Local Windows CPU silent-dies at chemprop epoch 14 (v3+v4 both). Kaggle T4 is the workaround.

Trains 2 single-task MPNNs (main pec50 + null pec50) on augmented corpus:
- 4139 original TRAIN (w=1.0)
- 95 semi-pure (drop 1 leaked) (w=0.7)
- 456 crudes (w=0.4; x0.5 if CAD_CV>5%)
- 253 phase1_unblinded HELD OUT (honest val)

Architecture: MPNN(depth=3, d_h=300) + FFN(2 layers, dropout=0.1).
Outputs to /kaggle/working/: te_chemprop_aux_v2.npy, ph_chemprop_aux_v2.npy, nb950_summary.json, nb950_chemprop_aux_v2.csv

In [ ]:
import subprocess, sys, os, time, json, traceback
os.environ['PYTHONUNBUFFERED'] = '1'
# v9 (Option D): ABANDON chemprop on Kaggle. v8 confirmed chemprop 2.0.4 AND
# 2.1.2 BOTH transitively pull numpy 1.26.4 which collides with Kaggle's
# numpy-2.x scipy build. Pure LGBM fallback path: LGBM + RDKit only (both
# Kaggle-stable since rdkit ships preinstalled on Kaggle's base image and
# lightgbm is numpy-2 compatible).
# v9b fix: drop rdkit-pypi (no longer on PyPI as a separate distro; Kaggle
# bundles rdkit anyway). Install only lightgbm; try-import rdkit and only
# fall back to pip if missing.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'], check=False)
try:
    import rdkit
    print(f'rdkit preinstalled: {rdkit.__version__}')
except ImportError:
    print('rdkit not preinstalled; installing rdkit (PyPI distro)...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rdkit'], check=False)
    import rdkit
    print(f'rdkit installed: {rdkit.__version__}')

import numpy, scipy
print(f'numpy: {numpy.__version__}  scipy: {scipy.__version__}')

import sklearn as _sk
import lightgbm as lgb
print(f'sklearn: {_sk.__version__}  lightgbm: {lgb.__version__}')

# No GPU needed (LGBM CPU is faster than GPU on this scale)
ACCEL = 'cpu'
print(f'ACCEL = {ACCEL}  (LGBM CPU path -- chemprop abandoned per cycle 162)')

In [ ]:
# Fetch all augmented corpus CSVs from HuggingFace
import urllib.request
from pathlib import Path

HF = 'https://huggingface.co/datasets/openadmet/pxr-challenge-train-test/resolve/main'
DATA = Path('/kaggle/working/rawdata'); DATA.mkdir(exist_ok=True, parents=True)

FILES = {
    'train':     'pxr-challenge_TRAIN.csv',
    'test':      'pxr-challenge_TEST_BLINDED.csv',
    'counter':   'pxr-challenge_counter-assay_TRAIN.csv',
    'semi_pure': 'pxr-challenge_96-compound-uscale-semi-pure_TRAIN.csv',
    'crudes':    'pxr-challenge_htchem-libraries_TRAIN.csv',
    'phase1':    'pxr-challenge_TEST_PHASE_1_UNBLINDED.csv',
}
for k, fn in FILES.items():
    p = DATA / fn
    if not p.exists():
        urllib.request.urlretrieve(f'{HF}/{fn}', p)
    print(f'{k}: {p.name} ({p.stat().st_size:,} bytes)')

In [ ]:
# Inline cheminformatics helpers (no dep on src/pxr)
import numpy as np, pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

_LE = rdMolStandardize.LargestFragmentChooser()
_UN = rdMolStandardize.Uncharger()

def standardize_smiles(smi):
    if smi is None or (isinstance(smi, float) and pd.isna(smi)): return None
    try:
        m = Chem.MolFromSmiles(str(smi))
        if m is None: return None
        m = _LE.choose(m); m = _UN.uncharge(m)
        return Chem.MolToSmiles(m, canonical=True)
    except Exception:
        return None

def to_inchikey(smi):
    if smi is None: return None
    try:
        m = Chem.MolFromSmiles(str(smi))
        return Chem.MolToInchiKey(m) if m is not None else None
    except Exception:
        return None

def bemis_murcko(smi):
    if smi is None: return ''
    try:
        m = Chem.MolFromSmiles(str(smi))
        if m is None: return ''
        scaf = MurckoScaffold.GetScaffoldForMol(m)
        return Chem.MolToSmiles(scaf, canonical=True)
    except Exception:
        return ''

def rae(yt, yp):
    yt = np.asarray(yt, float); yp = np.asarray(yp, float)
    return float(np.mean(np.abs(yt-yp)) / np.mean(np.abs(yt-yt.mean())))

from collections import defaultdict
def scaffold_kfold_indices(scaffolds, n_splits=5, seed=42):
    s2idx = defaultdict(list)
    for i, s in enumerate(scaffolds): s2idx[s].append(i)
    buckets = sorted(s2idx.values(), key=len, reverse=True)
    folds = [[] for _ in range(n_splits)]; sizes = [0]*n_splits
    for b in buckets:
        f = min(range(n_splits), key=lambda k: sizes[k])
        folds[f].extend(b); sizes[f] += len(b)
    all_idx = list(range(len(scaffolds)))
    return [(sorted(set(all_idx)-set(folds[k])), folds[k]) for k in range(n_splits)]

print('cheminf helpers OK')

In [ ]:
# Build augmented corpus matching scripts/nb950_chemprop_aux_v2.py:build_corpus
# v2 attempt #2: drop tier downweighting (semi_pure/crudes at 1.0) to fight variance
# compression. Cycle 141 v2 phase1=0.9588 with pred_std=0.617 (truth_std 1.03);
# heavy downweight + early-stop p=8 collapsed predictions. Per cycle 141 diagnosis,
# letting noisier tiers contribute at full weight should restore variance.
SEED = 42
TIER_W = {'original': 1.0, 'semi_pure': 1.0, 'crudes': 1.0}
CAD_CV_PENALTY = 1.0
_NUMERIC_COLS = ('pec50', 'pec50_null', 'pec50_se', 'emax', 'emax_se')
_CRUDE_NUMERIC_COLS = (
    'Crude CAD Peak Area CV (%)', 'Crude Product Yield (%)',
    'Crude Correction Factor', 'Crude CAD Slope CV (%)', 'Crude CAD Yield SE (log10)',
)

# Same rename maps as src/pxr/data.py
_DR_RENAME = {
    'Molecule Name':'name','SMILES':'smiles','OCNT_ID':'ocnt_id',
    'pEC50':'pec50','pEC50_std.error (-log10(molarity))':'pec50_se',
    'Emax_estimate (log2FC vs. baseline)':'emax',
    'Emax_std.error (log2FC vs. baseline)':'emax_se',
}
_SP_RENAME = {
    'SMILES':'smiles','OCNT_ID':'ocnt_id','Semi-Pure Batch ID':'batch',
    'Corrected Semi-Pure pEC50 (log)':'pec50',
    'Corrected Semi-Pure pEC50 ±1 SE (log)':'pec50_se',
}
_CR_RENAME = {
    'SMILES':'smiles','OCNT_ID':'ocnt_id','Crude Batch ID':'batch',
    'Corrected Crude pEC50 (log)':'pec50',
    'Corrected Crude pEC50 ±1 SE (log)':'pec50_se',
}
_TE_RENAME = {'Molecule Name':'name','SMILES':'smiles'}

def _rn(df, m): return df.rename(columns={k:v for k,v in m.items() if k in df.columns})

tr = _rn(pd.read_csv(DATA / FILES['train']), _DR_RENAME)
sp = _rn(pd.read_csv(DATA / FILES['semi_pure']), _SP_RENAME)
cr = _rn(pd.read_csv(DATA / FILES['crudes']), _CR_RENAME)
co = _rn(pd.read_csv(DATA / FILES['counter']), _DR_RENAME)
te = _rn(pd.read_csv(DATA / FILES['test']), _TE_RENAME)
ph = _rn(pd.read_csv(DATA / FILES['phase1']), _DR_RENAME)

raw_shapes = {'train':tr.shape,'semi_pure':sp.shape,'crudes':cr.shape,
              'counter':co.shape,'test':te.shape,'phase1':ph.shape}
print('raw shapes:', raw_shapes)
assert tr.shape[0]==4139 and co.shape[0]==2859 and te.shape[0]==513
assert ph.shape[0]==253 and sp.shape[0]==96 and cr.shape[0]==456

def coerce(df, cols):
    for c in cols:
        if c in df.columns: df[c] = pd.to_numeric(df[c], errors='coerce')

for d in (tr, sp, cr, co, ph):
    coerce(d, _NUMERIC_COLS)
coerce(cr, _CRUDE_NUMERIC_COLS)

drop_log = {}
for name, df in (('train',tr),('semi_pure',sp),('crudes',cr),('counter',co),('phase1',ph)):
    if 'pec50' in df.columns:
        bad = df['pec50'].isna()
        drop_log[name] = int(bad.sum())
        df.drop(df.index[bad], inplace=True)
print('drop_log =', drop_log)

for df in (tr, sp, cr, co, te, ph):
    df['std_smiles'] = df['smiles'].apply(standardize_smiles)
    df.dropna(subset=['std_smiles'], inplace=True)

te['inchikey'] = te['std_smiles'].apply(to_inchikey)
te_ikeys = set(te['inchikey'].dropna())

tr_main = tr.loc[tr['pec50'].notna(), ['std_smiles','pec50']].copy()
tr_main['weight'] = TIER_W['original']; tr_main['source'] = 'original'

sp['inchikey'] = sp['std_smiles'].apply(to_inchikey)
leaked = sp['inchikey'].isin(te_ikeys)
sp_clean = sp.loc[~leaked & sp['pec50'].notna(), ['std_smiles','pec50']].copy()
sp_clean['weight'] = TIER_W['semi_pure']; sp_clean['source'] = 'semi_pure'
print(f'semi_pure: kept {len(sp_clean)} / {len(sp)} (leaked={int(leaked.sum())})')

cad_cv = cr.get('Crude CAD Peak Area CV (%)')
cr2 = cr.loc[cr['pec50'].notna(), ['std_smiles','pec50']].copy()
cr2 = cr2.assign(cad_cv=cad_cv.reindex(cr2.index).values if cad_cv is not None else np.nan)
cr_w = np.full(len(cr2), TIER_W['crudes'], dtype=np.float32)
high_cv = (cr2['cad_cv'].values > 5) & cr2['cad_cv'].notna().values
cr_w[high_cv] *= CAD_CV_PENALTY
cr2['weight'] = cr_w; cr2['source'] = 'crudes'
cr2 = cr2.drop(columns=['cad_cv'])
print(f'crudes: kept {len(cr2)} rows; high-CV downweighted {int(high_cv.sum())}')

co_clean = co.loc[co['pec50'].notna(), ['std_smiles','pec50']].copy()
co_clean = co_clean.rename(columns={'pec50':'pec50_null'})
co_grouped = co_clean.groupby('std_smiles', as_index=False)['pec50_null'].median()

corpus = pd.concat([tr_main, sp_clean, cr2], ignore_index=True)
corpus = corpus.sort_values('weight', ascending=False)
corpus = corpus.drop_duplicates(subset=['std_smiles'], keep='first').reset_index(drop=True)
corpus = corpus.merge(co_grouped, on='std_smiles', how='left')
corpus['scaffold'] = corpus['std_smiles'].apply(bemis_murcko).fillna('')

print(f'FINAL corpus: {len(corpus)} unique SMILES')
print(corpus['source'].value_counts().to_dict())
print(f'null head non-null: {corpus["pec50_null"].notna().sum()}')

te_out = te[['std_smiles','name']].drop_duplicates(subset=['std_smiles']).reset_index(drop=True)
ph_out = ph.loc[ph['pec50'].notna(), ['std_smiles','name','pec50']].reset_index(drop=True)
print(f'test-513 std: {len(te_out)}  phase1: {len(ph_out)}')

In [ ]:
# v9 Option D: Pure LGBM on combined features (Morgan 2048 + RDKit descriptors).
# Output names KEPT as te_chemprop_aux_v2.npy / ph_chemprop_aux_v2.npy / nb950_chemprop_aux_v2.csv
# so downstream ladder/eval pipeline still picks up the artifact under the
# canonical name (cycle 162 directive: known-baseline reference under same name).
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(name)s] %(levelname)s: %(message)s')

from lightgbm import LGBMRegressor
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors

SEED = 42
t_start = time.time()
summary = {
    'script':'nb950_chemprop_aux_v2_kaggle_option_D',
    'tier_weights': TIER_W, 'cad_cv_penalty': CAD_CV_PENALTY,
    'corpus_total': int(len(corpus)),
    'corpus_by_source': corpus['source'].value_counts().to_dict(),
    'n_test_513': int(len(te_out)), 'n_phase1_unblinded': int(len(ph_out)),
    'null_head_n_labels': int(corpus['pec50_null'].notna().sum()),
    'leaked_sp_dropped': 1, 'dirty_rows_dropped': drop_log,
    'fallback_used': False, 'model':'lgbm_combined_v9_optionD', 'accel': ACCEL,
    'chemprop_abandoned_reason': 'chemprop 2.0.4/2.1.2 BOTH pull numpy 1.26.4, breaking Kaggle scipy ABI (cycle 162)',
}

# --- Feature builders ---
def morgan_fp(smi_list, n_bits=2048):
    out = np.zeros((len(smi_list), n_bits), dtype=np.float32)
    for i, s in enumerate(smi_list):
        m = Chem.MolFromSmiles(str(s))
        if m is not None:
            fp = AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=n_bits)
            out[i] = np.array(fp, dtype=np.float32)
    return out

_DESC_NAMES = [n for n, _ in Descriptors._descList]
_DESC_CALC = MoleculeDescriptors.MolecularDescriptorCalculator(_DESC_NAMES)
def rdkit_desc(smi_list):
    out = np.full((len(smi_list), len(_DESC_NAMES)), np.nan, dtype=np.float32)
    for i, s in enumerate(smi_list):
        m = Chem.MolFromSmiles(str(s))
        if m is not None:
            try:
                vals = _DESC_CALC.CalcDescriptors(m)
                out[i] = np.array(vals, dtype=np.float32)
            except Exception:
                pass
    return out

def combined_features(smi_list):
    mfp = morgan_fp(smi_list)
    rdd = rdkit_desc(smi_list)
    X = np.concatenate([mfp, rdd], axis=1).astype(np.float32)
    # Median-impute NaN/inf
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return X

print(f'[features] building combined (Morgan 2048 + RDKit {len(_DESC_NAMES)})')
t0 = time.time()
X_tr = combined_features(corpus['std_smiles'].tolist())
X_te = combined_features(te_out['std_smiles'].tolist())
X_ph = combined_features(ph_out['std_smiles'].tolist())
print(f'  X_tr: {X_tr.shape}  X_te: {X_te.shape}  X_ph: {X_ph.shape}  ({time.time()-t0:.1f}s)')

y_tr = corpus['pec50'].to_numpy(dtype=np.float32)
w_tr = corpus['weight'].to_numpy(dtype=np.float32)

# --- Main pec50 head (LGBM combined) ---
print('[main_pec50] training LGBM(n=500, leaves=64, lr=0.05)')
t_main = time.time()
model_main = LGBMRegressor(
    n_estimators=500, num_leaves=64, learning_rate=0.05,
    n_jobs=-1, random_state=SEED, verbose=-1,
)
model_main.fit(X_tr, y_tr, sample_weight=w_tr)
fit_main_s = time.time() - t_main
print(f'  done in {fit_main_s/60:.2f} min')

# --- Null pec50 head (LGBM on non-NaN subset) ---
null_mask = ~np.isnan(corpus['pec50_null'].to_numpy(dtype=np.float32))
n_null = int(null_mask.sum())
print(f'[null_pec50] training on {n_null}/{len(corpus)} non-NaN rows')
use_null = n_null >= 500
fit_null_s = 0.0
if use_null:
    t_null = time.time()
    X_null = X_tr[null_mask]
    y_null = corpus['pec50_null'].to_numpy(dtype=np.float32)[null_mask]
    w_null = w_tr[null_mask]
    model_null = LGBMRegressor(
        n_estimators=500, num_leaves=64, learning_rate=0.05,
        n_jobs=-1, random_state=SEED, verbose=-1,
    )
    model_null.fit(X_null, y_null, sample_weight=w_null)
    fit_null_s = time.time() - t_null
    print(f'  done in {fit_null_s/60:.2f} min')

# --- Predict + 0.70/0.30 main/null blend (same recipe as chemprop_aux v1) ---
te_main = model_main.predict(X_te)
ph_main = model_main.predict(X_ph)
if use_null:
    te_null = model_null.predict(X_te)
    ph_null = model_null.predict(X_ph)
    te_preds = 0.70 * te_main + 0.30 * te_null
    ph_preds = 0.70 * ph_main + 0.30 * ph_null
else:
    te_preds, ph_preds = te_main, ph_main

lo = float(np.nanmin(y_tr)) - 0.5
hi = float(np.nanmax(y_tr)) + 0.5
te_preds = np.clip(te_preds, lo, hi).astype(np.float32)
ph_preds = np.clip(ph_preds, lo, hi).astype(np.float32)

fallback = False  # not a fallback path; this IS the deliberate model
summary.update({
    'fit_main_minutes': fit_main_s/60,
    'fit_null_minutes': fit_null_s/60,
    'null_head_used': bool(use_null),
    'null_head_n_rows': int(n_null),
    'n_features': int(X_tr.shape[1]),
    'feature_set': 'morgan2048+rdkit_desc',
    'blend_weights': {'main': 0.70, 'null': 0.30},
})

print(f'te_preds: shape={te_preds.shape} mean={te_preds.mean():.3f} std={te_preds.std():.3f}')
print(f'ph_preds: shape={ph_preds.shape} mean={ph_preds.mean():.3f} std={ph_preds.std():.3f}')

In [ ]:
# Save all artefacts to /kaggle/working
from pathlib import Path
WORK = Path('/kaggle/working')

if fallback:
    te_name = 'te_nb950b_lgbm_v2.npy'
    ph_name = 'ph_nb950b_lgbm_v2.npy'
    sub_name = 'nb950b_lgbm_v2.csv'
else:
    te_name = 'te_chemprop_aux_v2.npy'
    ph_name = 'ph_chemprop_aux_v2.npy'
    sub_name = 'nb950_chemprop_aux_v2.csv'

np.save(WORK / te_name, te_preds.astype(np.float32))
np.save(WORK / ph_name, ph_preds.astype(np.float32))
print(f'saved {te_name}  shape={te_preds.shape}')
print(f'saved {ph_name}  shape={ph_preds.shape}')

# Submission CSV in challenge format (SMILES + Molecule Name + pEC50), ordered to raw test-513
raw_test = _rn(pd.read_csv(DATA / FILES['test']), _TE_RENAME)
raw_test['std_smiles'] = raw_test['smiles'].apply(standardize_smiles)
pred_map = dict(zip(te_out['std_smiles'].tolist(), te_preds.tolist()))
raw_test['pEC50'] = raw_test['std_smiles'].map(pred_map)
if raw_test['pEC50'].isna().any():
    fill = float(corpus['pec50'].mean())
    n_miss = int(raw_test['pEC50'].isna().sum())
    print(f'WARNING: {n_miss} test rows missing prediction; filling with training mean {fill:.3f}')
    raw_test['pEC50'] = raw_test['pEC50'].fillna(fill)
sub = raw_test.rename(columns={'smiles':'SMILES','name':'Molecule Name'})[['SMILES','Molecule Name','pEC50']]
sub.to_csv(WORK / sub_name, index=False)
print(f'saved submission: {sub_name}  ({len(sub)} rows)')

# Honest validation on phase1_unblinded (253 held out)
y_ph_true = ph_out['pec50'].to_numpy(dtype=np.float32)
ph_rae = rae(y_ph_true, ph_preds)
summary['phase1_unblinded_RAE'] = float(ph_rae)
summary['phase1_unblinded_n'] = int(len(ph_out))
summary['phase1_unblinded_pred_mean'] = float(ph_preds.mean())
summary['phase1_unblinded_pred_std'] = float(ph_preds.std())
summary['phase1_unblinded_true_mean'] = float(y_ph_true.mean())
summary['phase1_unblinded_true_std'] = float(y_ph_true.std())
summary['wall_time_min'] = (time.time() - t_start) / 60

with open(WORK / 'nb950_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print('saved nb950_summary.json')

print('='*70)
print(f'phase1_unblinded RAE: {ph_rae:.4f}  (target <= 0.6216 to beat chemprop_aux v1)')
print(f'  improvement vs 0.6216: {0.6216 - ph_rae:+.4f}')
print(f'  wall time: {summary["wall_time_min"]:.1f} min  fallback={fallback}')
print('='*70)